# Local Parquet Explorer

Query materialized Dagster assets stored as local parquet files in `data/`.

**Prerequisites:** Run `make install` then materialize assets via the Dagster UI (`make dev`).

In [ ]:
# ── Setup ────────────────────────────────────────────────
from pathlib import Path

import polars as pl
from IPython.display import display, HTML

DATA_ROOT = Path("../data")
CLEAN = DATA_ROOT / "clean"
LANDING = DATA_ROOT / "landing"

_con = None

def _duckdb():
    """Lazy DuckDB connection — only created on first sql()/register() call."""
    global _con
    if _con is None:
        import duckdb
        _con = duckdb.connect()
    return _con


# ── Display helper ────────────────────────────────────────
def show(df: pl.DataFrame, max_rows: int = 200):
    """Scrollable HTML table with sticky headers."""
    pdf = df.head(max_rows).to_pandas()
    table_html = pdf.to_html(index=False, border=0, classes=["dataframe"])
    display(HTML(f"""
    <div style="max-height:500px; overflow-y:auto; overflow-x:auto; border:1px solid #444;">
        <style>
            .dataframe thead th {{ position:sticky; top:0; background:#222; color:white; z-index:1; }}
            .dataframe td {{ white-space:nowrap; font-size:13px; }}
        </style>
        {table_html}
    </div>
    <p style="color:#888; font-size:12px;">{df.height} rows x {df.width} cols{f' (showing {max_rows})' if df.height > max_rows else ''}</p>
    """))


# ── Convenience functions ─────────────────────────────────
def scan(asset_name: str) -> pl.LazyFrame:
    """Scan a clean asset's parquet files (handles partitioned and single-file)."""
    asset_dir = CLEAN / asset_name
    single = asset_dir / f"{asset_name}.parquet"
    if single.exists():
        return pl.scan_parquet(single)
    return pl.scan_parquet(str(asset_dir / "**/*.parquet"), hive_partitioning=True)


def load(asset_name: str) -> pl.DataFrame:
    """Load a clean asset into memory."""
    return scan(asset_name).collect()


def sql(query: str) -> pl.DataFrame:
    """Run SQL against registered DuckDB views."""
    return _duckdb().execute(query).pl()


def register(asset_name: str, table_name: str | None = None):
    """Register a clean asset as a DuckDB view for SQL queries."""
    name = table_name or asset_name
    asset_dir = CLEAN / asset_name
    single = asset_dir / f"{asset_name}.parquet"
    if single.exists():
        path = str(single)
    else:
        path = str(asset_dir / "**/*.parquet")
    _duckdb().execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}', hive_partitioning=true, union_by_name=true)")
    print(f"Registered: {name}")


def catalog():
    """List all materialized assets in data/clean/."""
    if not CLEAN.exists():
        print("No data/clean/ directory. Run 'make dev' and materialize some assets first.")
        return
    assets = []
    for d in sorted(CLEAN.iterdir()):
        if not d.is_dir():
            continue
        parquets = list(d.rglob("*.parquet"))
        if not parquets:
            continue
        total_bytes = sum(p.stat().st_size for p in parquets)
        size_mb = round(total_bytes / 1_048_576, 2)
        assets.append({"asset": d.name, "files": len(parquets), "size_mb": size_mb})
    if not assets:
        print("No materialized assets found. Materialize via the Dagster UI first.")
        return
    show(pl.DataFrame(assets))


def describe(asset_name: str):
    """Show schema of a materialized asset."""
    lf = scan(asset_name)
    schema = lf.collect_schema()
    rows = [{"column": name, "type": str(dtype)} for name, dtype in schema.items()]
    print(f"Schema: {asset_name} ({len(rows)} columns)")
    show(pl.DataFrame(rows))


def export(df: pl.DataFrame, name: str, fmt: str = "csv"):
    """Export a DataFrame to data/exports/."""
    out = DATA_ROOT / "exports" / f"{name}.{fmt}"
    out.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "parquet":
        df.write_parquet(str(out))
    elif fmt == "csv":
        df.write_csv(str(out))
    elif fmt == "json":
        df.write_ndjson(str(out))
    print(f"Exported {df.height} rows to {out}")


print("Ready. Use catalog(), describe(), scan(), load(), sql(), register(), show(), export().")

## Browse Materialized Assets

In [ ]:
catalog()

## Describe an Asset

In [ ]:
describe("nyc_floodnet_events_joined")

In [ ]:
describe("nyc_film_permits")

## Query with Polars

In [ ]:
# FloodNet sensors by borough
df = (
    scan("nyc_floodnet_sensor_metadata")
    .group_by("borough")
    .agg(pl.len().alias("sensors"))
    .sort("sensors", descending=True)
    .collect()
)
show(df)

In [ ]:
# Flooding events — top 10 sensors by event count
df = (
    scan("nyc_floodnet_flooding_events")
    .group_by("sensor_name")
    .agg(
        pl.len().alias("events"),
        pl.col("max_depth_inches").max().alias("worst_depth_in"),
    )
    .sort("events", descending=True)
    .head(10)
    .collect()
)
show(df)

In [ ]:
# Joined events — severity breakdown
df = (
    scan("nyc_floodnet_events_joined")
    .group_by("flood_severity")
    .agg(
        pl.len().alias("events"),
        pl.col("max_depth_inches").mean().round(1).alias("avg_depth_in"),
        pl.col("duration_mins").mean().round(1).alias("avg_duration_min"),
    )
    .sort("events", descending=True)
    .collect()
)
show(df)

## Query with DuckDB SQL

Register assets as DuckDB views, then use SQL.

In [ ]:
register("nyc_floodnet_sensor_metadata", "sensors")
register("nyc_floodnet_flooding_events", "events")
register("nyc_floodnet_events_joined", "joined")

In [ ]:
# Sensors with most flooding events
df = sql("""
SELECT s.sensor_name, s.borough, COUNT(e.sensor_id) AS events
FROM sensors s
LEFT JOIN events e ON s.sensor_id = e.sensor_id
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 15
""")
show(df)

In [ ]:
# Severity by borough from joined asset
df = sql("""
SELECT borough, flood_severity, COUNT(*) AS events,
       ROUND(AVG(max_depth_inches), 1) AS avg_depth,
       ROUND(AVG(duration_mins), 1) AS avg_duration
FROM joined
GROUP BY 1, 2
ORDER BY 1, 4 DESC
""")
show(df)

In [ ]:
# Hydro metrics — flashiest floods (lowest time_to_peak_pct)
df = sql("""
SELECT sensor_name, borough, flood_start_time,
       max_depth_inches, duration_mins,
       ROUND(rise_rate_in_per_min, 3) AS rise_rate,
       ROUND(time_to_peak_pct, 3) AS peak_pct
FROM joined
WHERE time_to_peak_pct IS NOT NULL
ORDER BY time_to_peak_pct ASC
LIMIT 20
""")
show(df)

## 311 Sample (partitioned)

In [ ]:
describe("nyc_311_sample")

In [ ]:
df = (
    scan("nyc_311_sample")
    .group_by("complaint_type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
    .collect()
)
show(df)

## Film Permits

In [ ]:
describe("nyc_film_permits")

In [ ]:
df = (
    scan("nyc_film_permits")
    .group_by("borough", "category")
    .agg(pl.len().alias("permits"))
    .sort("permits", descending=True)
    .head(15)
    .collect()
)
show(df)

---
## Your Queries

Use `scan()` for lazy Polars, `load()` for eager, `sql()` for DuckDB, `show()` to render.

In [ ]:
# Your query here
# df = scan("nyc_floodnet_events_joined").head(10).collect()
# show(df)

## Export Results

Export any DataFrame to `data/exports/` as CSV, Parquet, or JSON.

In [ ]:
# export(df, "my_export", "csv")
# export(df, "my_export", "parquet")